# Clase 6 - Bases de Datos

Actividades:
1. Elijan y carguen un dataset desde CSV en pandas.
2. Analicen si la tabla está normalizada y en qué nivel (1FN, 2FN, 3FN).
3. Generen el conjunto de tablas para alcanzar el nivel 3FN. Asignen las claves primarias en cada tabla.
4. Guarden el conjunto como una base de datos SQLite.
5. Generen una consulta a la base de datos.

## 1. Cargo el dataset desde CSV en pandas

In [8]:
import sqlite3
import pandas as pd


In [14]:
df = pd.read_csv("data/sleep_mobile_stress_dataset_15000.csv")

## 2. Analizo si la tabla esta normalizada

In [ ]:
# 1FN: Valores atomicos, sin filas duplicadas ni celdas multivaluadas
print(df.dtypes)
print("\nFilas duplicadas:", df.duplicated().sum())
print("Valores nulos por columna:\n", df.isnull().sum())

user_id                               int64
age                                   int64
gender                                  str
occupation                              str
daily_screen_time_hours             float64
phone_usage_before_sleep_minutes      int64
sleep_duration_hours                float64
sleep_quality_score                 float64
stress_level                        float64
caffeine_intake_cups                  int64
physical_activity_minutes             int64
notifications_received_per_day        int64
mental_fatigue_score                float64
dtype: object

Filas duplicadas: 0
Valores nulos por columna:
 user_id                             0
age                                 0
gender                              0
occupation                          0
daily_screen_time_hours             0
phone_usage_before_sleep_minutes    0
sleep_duration_hours                0
sleep_quality_score                 0
stress_level                        0
caffeine_intake_cups   

In [12]:
# 2FN: La clave primaria debe ser simple (no compuesta) para que aplique automaticamente
print("user_id es unico:", df["user_id"].is_unique)

user_id es unico: True


In [13]:
# 3FN: busco dependencias transitivas entre atributos no-clave

columnas_categoricas = ["gender", "occupation"]
otras_columnas = [c for c in df.columns if c not in columnas_categoricas + ["user_id"]]

for cat in columnas_categoricas:
    print(f"\n--- Buscando dependencias funcionales desde '{cat}' ({df[cat].nunique()} valores unicos) ---")
    encontro_alguna = False
    for col in otras_columnas:
        valores_unicos_por_grupo = df.groupby(cat)[col].nunique()
        if (valores_unicos_por_grupo == 1).all():
            print(f"  POSIBLE dependencia funcional: {cat} -> {col}")
            encontro_alguna = True
    if not encontro_alguna:
        print(f"  No se encontraron dependencias funcionales desde '{cat}'.")


--- Buscando dependencias funcionales desde 'gender' (3 valores unicos) ---
  No se encontraron dependencias funcionales desde 'gender'.

--- Buscando dependencias funcionales desde 'occupation' (8 valores unicos) ---
  No se encontraron dependencias funcionales desde 'occupation'.


> Idea: si TODAS las filas con el mismo valor de una columna categorica (ej. "occupation") comparten siempre el mismo valor en otra columna, hay una dependencia funcional categoria -> otra_columna que NO depende de user_id, sino de otro atributo no-clave. Eso violaria la 3FN.

**Conclusion del analisis de normalizacion:**

- **1FN** se cumple. Todas las columnas tienen valores atomicos (números o texto simple, sin listas), no hay filas duplicadas y no hay valores nulos.
- **2F:** se cumple. La clave primaria (`user_id`) es una única columna, por lo que no puede haber dependencias parciales (esas sólo se dan con claves compuestas).
- **3FN** se cumple. Al probar dependencias funcionales entre las columnas categoricas (`gender`, `occupation`) y el resto de las columnas, no aparecio ningun caso donde un valor de `gender` u `occupation` determine siempre el mismo valor en otra columna. Es decir, todos los atributos (edad, tiempo de pantalla, calidad de sueño, estres, etc.) dependen directamente de `user_id` y no de otro atributo no-clave.

## 3. Generen las tablas para alcanzar el nivel 3FN y asignar claves primarias en cada tabla

La tabla `sleepTable` ya esta en **3FN**. Cada fila representa a un único usuario (`user_id`), y todas las columnas son características propias de ese usuario, sin dependencias transitivas ni redundancias evidentes. Por eso, para este punto de la práctica, no haría falta descomponer esta tabla en varias tablas adicionales: una unica tabla con clave primaria `user_id` ya cumple 3FN.

## 4. Guardo el conjunto como una base de datos SQLite.

In [20]:
# Conecto a una BBDD SQLite
conn = sqlite3.connect("sleep_database.db")
# Escribo los datos a una tabla SQL
df.to_sql("sleepTable", conn, if_exists="replace", index=False)

15000

## 5. Genero una consulta a la base de datos

In [21]:
# Consulta: promedio de horas de sueño y nivel de estres por ocupacion
query = """
SELECT occupation,
       COUNT(*) AS cantidad_de_sujetos,
       ROUND(AVG(sleep_duration_hours), 2) AS promedio_horas_de_sueno,
       ROUND(AVG(stress_level), 2) AS promedio_de_nivel_de_estres
FROM sleepTable
GROUP BY occupation
ORDER BY promedio_de_nivel_de_estres DESC;
"""

resultado = pd.read_sql_query(query, conn)
conn.close()

resultado

,occupation,cantidad_de_sujetos,promedio_horas_de_sueno,promedio_de_nivel_de_estres
0,Student,1861,6.53,7.18
1,Manager,1962,6.52,7.06
2,Software Engineer,1917,6.51,7.02
3,Doctor,1922,6.45,6.97
4,Researcher,1837,6.50,6.92
5,Teacher,1797,6.54,6.90
6,Freelancer,1891,6.49,6.90
7,Designer,1813,6.54,6.87
